# Assignment 3: Reconstruct, evaluate, and export

A3 adds inference only. It imports A1's pipeline and A2's model/training
modules, reconstructs exact saved architectures, verifies hashes, and uses
held-out test loaders that were never used for selection.


## Grading rubric (100 points)

| Assessed evidence | Points | Full-credit standard |
|---|---:|---|
| Provenance and exact reconstruction | 15 | A1 protocol, A2 source, constructor configuration, and all checkpoint hashes verified before inference |
| Held-out MNIST inference | 10 | Exact A1 test membership; exact A2 ResNet checkpoint; every batch; confusion/per-class evidence |
| Held-out EuroSAT inference | 10 | Exact A1 test membership; exact A2 classifier checkpoint; every batch and per-class support |
| Planet inference and GeoTIFF export | 25 | Three-class argmax; every test chip; global metrics; class 2 retained; CRS/transform/grid/dtype/nodata independently verified |
| Failure analysis and model card | 25 | Predetermined ranking; weak and median cases; per-class interpretation; limitations, provenance and prohibited uses documented |
| Critical understanding answers | 15 | Specific reasoning about leakage, undefined metrics, spatial dependence, nodata, accuracy and failure cases |
| **Total** | **100** | |

Inference credit requires a frozen model and test set. Retraining, changing
preprocessing, or selecting a checkpoint after inspecting test outcomes invalidates
the affected evaluation evidence.


## Where you write code in the modular assignment

The notebooks are the **orchestrators**, not the main implementation files. Keep
them open to read instructions, reload your modules, run checks, make figures, and
write interpretations. Write assessed Python implementations in these exact files:

| Assignment | File you edit | Symbols you implement |
|---|---|---|
| A1 | `src/student_pipeline.py` | `paired_flip`, `normalize_image`, `training_transform`, `evaluation_transform`, `build_mnist_loaders`, `build_eurosat_loaders`, `PlanetCatalogDataset`, `match_directory_pairs`, `PlanetDirectoryDataset`, `build_planet_loaders`, `save_data_protocol`, `load_data_protocol` |
| A2 | `src/student_models.py` | `MNISTResNet18`, `EuroSATClassifier`, `PlanetUNet` |
| A2 | `src/student_training.py` | `run_epoch`, `fit` |
| A3 | `src/student_inference.py` | `predict_classification`, `export_prediction`, `predict_and_export` |

Replace the `TODO`/`NotImplementedError` bodies while preserving the public
function names, arguments, return contracts, and class attributes described in
their docstrings. Save the module, rerun the notebook's reload cell, and recreate
any datasets, loaders, models, or optimizers that were built from the old code.

You also write explanatory work in the notebook's clearly labelled evidence,
architecture, interpretation, and critical-question Markdown cells, and document
agent assistance in `AI_USE.md`. Do **not** implement assessed work in notebook
scratch cells or edit `assignment_setup.py`, `student_checks.py`, `assessment_support.py`,
the catalog, or the tests. Those supplied files define setup, checks, and evidence
contracts and should remain unchanged.


## Foundations: inference is a provenance exercise

Loading a `.pth` file is not sufficient evidence that the intended model ran. A
`state_dict` stores named tensors, not the Python forward graph, preprocessing, or
data membership. A3 therefore rebuilds the saved constructor configuration, checks
source and checkpoint hashes, and reconstructs A1's exact test IDs before loading
weights. If any hash differs, stop and explain the discrepancy rather than editing
the manifest.

Inference uses `eval()` and `no_grad()` for every batch. For classification,
`argmax(dim=1)` converts ten logits to one class ID. For segmentation it converts
`[B,3,H,W]` to `[B,H,W]` while preserving boundary class 2. Metrics are accumulated
from one global confusion matrix so a small last batch or a chip with few valid
pixels is not accidentally weighted like a large batch.

Pixel accuracy alone is unsafe for imbalanced maps. Report class support, IoU,
Dice, precision, recall, and the confusion matrix. A zero denominator means the
metric is undefined, not automatically zero. Georeferenced output adds another
contract: prediction height/width, CRS, affine transform, dtype, class values, and
nodata must be verified after reopening the file. Image validity controls output
nodata; missing reference annotation controls only which pixels contribute to
evaluation.

Failure analysis is not a gallery of attractive examples. Freeze a ranking rule
before looking at images, include weak cases, compare interior and boundary errors,
and distinguish an observed failure from a causal claim. The model card records
what was run, where it may fail, and uses that are not justified by this exercise.


## Setup: upload the complete folder

**2i2c is the primary platform.** Upload or extract the entire folder as
`/home/jovyan/assignments`, then open this notebook from the folder root. Keep
`src/`, `catalog.csv`, all notebooks, and `outputs/` together.
The `/home/jovyan` directory is persistent across normal 2i2c sessions.

The setup below adds `PROJECT_ROOT/src` directly to Python's import path. It does
not create an environment or install the assignment as a package. It first checks
the modules already supplied by the 2i2c image. Only if that check reports a
missing package should you open a terminal in `assignments` and run:

```bash
python -m pip install -r requirements.txt
```

**Colab fallback:** put the complete folder in
`My Drive/IDLEO/assignments`, open the notebook in Colab, and run the same
cells. Code, protocols, outputs, and checkpoints remain in Drive. Runtime data
may need to be downloaded again after a reset.


In [ ]:
import importlib
import json  # noqa: F401
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/IDLEO/assignments")
else:
    preferred = Path("/home/jovyan/assignments")
    if preferred.is_dir():
        PROJECT_ROOT = preferred
    else:
        working_directory = Path.cwd().resolve()
        PROJECT_ROOT = working_directory

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from assignment_setup import check_dependencies, prepare_project  # noqa: E402

paths = prepare_project(PROJECT_ROOT)
versions = check_dependencies()
device_name = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("Project:", paths.project_root)
print("Data:", paths.data_root)
print("Device:", device_name)
print("Dependencies:", versions)

## Agentic coding without outsourcing the assignment

An agent may help you plan, inspect an error, propose a test, or review one saved
function. Work on one named TODO at a time. Give the function contract, shapes,
and the failing check; ask for an explanation of the proposed change. Read every
line, reject unsupported assumptions, save the file yourself, reload it, and run
both the supplied check and an independent counterexample. Do not ask an agent to
"complete the assignment" or paste an answer repository. Record what you accepted,
changed, rejected, and verified in `AI_USE.md`. You must be able to explain and
modify the final code without assistance.


## Reload the progressive source modules


In [ ]:
import pandas as pd
import torch

import student_checks as checks
import student_inference as inference
import student_models as models
import student_pipeline as pipeline
import student_training as training
from assessment_support import sha256, show_planet_triplet

pipeline = importlib.reload(pipeline)
models = importlib.reload(models)
training = importlib.reload(training)
inference = importlib.reload(inference)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 1: Verify provenance and rebuild the exact workflow (part of 15 points)

Load the A1 protocol and A2 manifest. Verify hashes before loading weights.
Reconstruct dataloaders from saved membership and models from saved constructor
dictionaries. Do not substitute a new architecture or new split.


In [ ]:
protocol_path = paths.output_root / "assignment_1/data_protocol.json"
manifest_path = paths.output_root / "assignment_2/experiment_manifest.json"
protocol = pipeline.load_data_protocol(protocol_path)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert sha256(protocol_path) == manifest["data_protocol_sha256"]
assert (
    sha256(paths.project_root / "src/student_pipeline.py")
    == manifest["student_pipeline_sha256"]
)
assert (
    sha256(paths.project_root / "src/student_models.py")
    == manifest["student_models_sha256"]
)
assert (
    sha256(paths.project_root / "src/student_training.py")
    == manifest["student_training_sha256"]
)

mnist = pipeline.build_mnist_loaders(
    paths.data_root / "mnist",
    batch_size=protocol["settings"]["mnist_batch_size"],
    seed=protocol["seed"],
    split_ids=protocol["mnist_split_ids"],
    download=True,
)
eurosat = pipeline.build_eurosat_loaders(
    paths.data_root / "eurosat",
    batch_size=protocol["settings"]["euro_batch_size"],
    seed=protocol["seed"],
    split_ids=protocol["eurosat_split_ids"],
    download=True,
)
planet = pipeline.build_planet_loaders(
    paths.catalog_path,
    paths.data_root,
    batch_size=protocol["settings"]["planet_batch_size"],
    seed=protocol["seed"],
    selected_rows=protocol["planet_selected_rows"],
)

## Task 2: Restore validation-selected checkpoints (part of the same 15 points)


In [ ]:
mnist_path = paths.output_root / "assignment_2" / manifest["mnist_checkpoint"]
classifier_path = (
    paths.output_root / "assignment_2" / manifest["classifier_checkpoint"]
)
segmenter_path = (
    paths.output_root / "assignment_2" / manifest["segmentation_checkpoint"]
)
assert sha256(mnist_path) == manifest["mnist_checkpoint_sha256"]
assert sha256(classifier_path) == manifest["classifier_checkpoint_sha256"]
assert sha256(segmenter_path) == manifest["segmentation_checkpoint_sha256"]

mnist_classifier = models.MNISTResNet18(**manifest["mnist_config"]).to(device)
mnist_state = torch.load(mnist_path, map_location=device)
mnist_classifier.load_state_dict(mnist_state["model_state"])

classifier = models.EuroSATClassifier(**manifest["classifier_config"]).to(
    device
)
classifier_state = torch.load(classifier_path, map_location=device)
classifier.load_state_dict(classifier_state["model_state"])

segmenter = models.PlanetUNet(**manifest["unet_config"]).to(device)
segmenter_state = torch.load(segmenter_path, map_location=device)
segmenter.load_state_dict(segmenter_state["model_state"])
checks.check_unet_architecture(segmenter.cpu())
segmenter = segmenter.to(device)

## Task 3: Held-out MNIST and EuroSAT inference (10 + 10 points)

Implement `predict_classification`. Process every test batch under
`eval()` and `no_grad()`, aggregate one confusion matrix, and report support
and per-class metrics. A3 may inspect test errors after model selection but
must not return to A2 and choose a different model from them.


In [ ]:
mnist_result = inference.predict_classification(
    mnist_classifier, mnist["test"], device
)
classification_result = inference.predict_classification(
    classifier, eurosat["test"], device
)
print("MNIST metrics:", mnist_result["metrics"])
print("EuroSAT metrics:", classification_result["metrics"])
display(
    pd.DataFrame(
        {
            "target": classification_result["targets"],
            "prediction": classification_result["predictions"],
        }
    )
)
print(classification_result["metrics"])

## Task 4: Three-class Planet inference and GeoTIFF export (25 points)

Implement `export_prediction` and `predict_and_export`. Use `argmax` across
three logits, preserve class `2`, and aggregate metrics across the entire
test split. Each output must preserve source CRS, transform, height and
width; use uint8 classes 0/1/2 and nodata 255 based on image validity—not
missing reference labels. Reopen and validate every exported file.


In [ ]:
checks.check_export(inference, paths.output_root / "assignment_3")
prediction_dir = paths.output_root / "assignment_3/predictions"
segmentation_result = inference.predict_and_export(
    segmenter, planet["test"], device, prediction_dir
)
print(segmentation_result["metrics"])
display(pd.DataFrame(segmentation_result["files"]))

sample_images, sample_targets, _ = next(iter(planet["test"]))
with torch.no_grad():
    sample_predictions = segmenter(sample_images.to(device)).argmax(1).cpu()
prediction_figure = show_planet_triplet(
    sample_images[0], sample_targets[0], sample_predictions[0]
)
prediction_figure.savefig(
    paths.output_root / "assignment_3/planet_prediction.png", dpi=150
)

## Task 5: Failure analysis and model card (25 points)

Before ranking examples, state a selection rule such as three lowest
per-chip macro-IoU cases plus one median case, with filename tie-breaking.
Compare noncrop, field-interior, and boundary behavior separately. Save a
`model_card.md` describing purpose, data populations, splits, preprocessing,
architecture, checkpoints, compute, metrics, limitations, and prohibited uses.

**Written answer:** add the rule, measured results, failure examples, and
model card here. Do not present only attractive maps.


In [ ]:
ranked_chips = sorted(
    segmentation_result["files"],
    key=lambda record: (record["metrics"]["mean_iou"], record["source_path"]),
)
failure_analysis = {
    "selection_rule": (
        "three lowest per-chip mean-IoU cases plus the median; "
        "filename breaks ties"
    ),
    "lowest_three": ranked_chips[:3],
    "median": ranked_chips[len(ranked_chips) // 2],
}
failure_path = paths.output_root / "assignment_3/failure_analysis.json"
failure_path.write_text(
    json.dumps(failure_analysis, indent=2), encoding="utf-8"
)
print(failure_path)

In [ ]:
MODEL_CARD = """# Model card: IDLEO 2026 reference models

## Intended purpose
TODO: describe the supported educational or research use and prohibited uses.

## Data and protocol
TODO: cite the saved A1 protocol, populations, split construction, and
preprocessing.

## Architecture and checkpoints
TODO: describe both architectures and record the exact checkpoint/source hashes.

## Evaluation
TODO: report aggregate and per-class metrics with denominators and uncertainty
caveats.

## Limitations and failure modes
TODO: discuss geographic dependence, boundary errors, coverage, and deployment
risks.
"""
model_card_path = paths.output_root / "assignment_3/model_card.md"
model_card_path.write_text(MODEL_CARD, encoding="utf-8")
print(model_card_path)

In [ ]:
results_path = paths.output_root / "assignment_3/evaluation_summary.json"
results_path.write_text(
    json.dumps(
        {
            "mnist_metrics": mnist_result["metrics"],
            "classification_metrics": classification_result["metrics"],
            "segmentation_metrics": segmentation_result["metrics"],
            "prediction_files": segmentation_result["files"],
            "failure_analysis": str(failure_path),
            "data_protocol_sha256": sha256(protocol_path),
            "experiment_manifest_sha256": sha256(manifest_path),
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(results_path)

## Critical understanding questions (15 points)

1. Which hashes prove that A3 used A1's protocol and A2's exact source/checkpoints?
2. Why does the MNIST test loader have to replay A1 membership instead of simply requesting any test subset?
3. Why is rebuilding the same class definition necessary before loading a `state_dict`?
4. What information would be leaked if A3 test scores were used to choose a new checkpoint?
5. Why should unsupported per-class metrics remain undefined rather than being replaced with zero?
6. How can global pixel accuracy remain high while boundary IoU is poor?
7. Why must GeoTIFF nodata follow image validity rather than reference-label availability?
8. Why does removing duplicate IDs not establish geographic independence?
9. How does a predetermined failure-example rule reduce presentation bias?
10. Which observed limitation should motivate the next experiment without changing this test report?
